<a href="https://colab.research.google.com/github/sanjoggrg1/Sanjog-TECH405/blob/main/Week7_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Sanjog Gurung
# TECH 405
# Wk7 Assignment BERT

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [ ]:
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
RAW_DATA = [

    # ───────────── POSITIVE ─────────────
    ("I absolutely love this product, it exceeded all my expectations!", "positive"),
    ("This is the best movie I have ever seen in my entire life.", "positive"),
    ("The customer service was outstanding and incredibly helpful.", "positive"),
    ("What a wonderful experience! I will definitely come back again.", "positive"),
    ("The food was delicious and the atmosphere was perfect.", "positive"),
    ("I am so happy with my purchase, highly recommend to everyone.", "positive"),
    ("This book changed my life, absolutely brilliant writing.", "positive"),
    ("The team did a fantastic job delivering the project on time.", "positive"),
    ("I had an amazing time at the concert, truly unforgettable.", "positive"),
    ("The hotel was luxurious and the staff were exceptionally kind.", "positive"),
    ("Great quality product, exactly what I was looking for!", "positive"),
    ("The app works flawlessly and the design is beautiful.", "positive"),
    ("Superb performance! The athlete broke records tonight.", "positive"),
    ("I feel so grateful for the support I received from the community.", "positive"),
    ("The sunset was breathtaking, one of the most beautiful sights ever.", "positive"),
    ("The doctor was compassionate and explained everything clearly.", "positive"),
    ("We had a fantastic road trip with no issues whatsoever.", "positive"),
    ("The workshop was incredibly insightful and well-organized.", "positive"),
    ("My kids absolutely loved the theme park, smiles all day long.", "positive"),
    ("This coffee tastes amazing, best brew I have had in years.", "positive"),
    ("The scholarship changed my entire academic journey for the better.", "positive"),
    ("I am thrilled with the results of my fitness journey so far.", "positive"),
    ("The new software update is a massive improvement over the last.", "positive"),
    ("Our wedding planner was exceptional, the day was perfect.", "positive"),
    ("The charity event raised more money than we ever hoped for.", "positive"),
    ("Brilliant storytelling that kept me on the edge of my seat.", "positive"),
    ("The professor made a complex subject easy and enjoyable.", "positive"),
    ("I love how responsive the support team is to every query.", "positive"),
    ("The new park in our neighborhood is a joy for the whole family.", "positive"),
    ("The training session was motivating and very well delivered.", "positive"),
    ("I was blown away by the quality of the artwork on display.", "positive"),
    ("Excellent value for money, I could not ask for more.", "positive"),
    ("The new policy has greatly benefited employees across the board.", "positive"),
    ("I feel energized and inspired after attending the conference.", "positive"),
    ("The garden looks absolutely stunning after the renovation.", "positive"),
    ("Outstanding research that will impact the field for years.", "positive"),
    ("The volunteer team worked tirelessly and achieved great results.", "positive"),
    ("I cannot believe how quickly the delivery arrived, impressive!", "positive"),
    ("The comedy show had us laughing the entire evening.", "positive"),
    ("A wonderful addition to the city that everyone can enjoy.", "positive"),
    ("The mentorship program has been transformative for my career.", "positive"),
    ("This recipe is a keeper, the whole family enjoyed it immensely.", "positive"),
    ("The new gym equipment is top-notch and very easy to use.", "positive"),
    ("I am proud of how far the students have come this semester.", "positive"),
    ("The documentary was eye-opening and beautifully produced.", "positive"),
    ("The new bus route has made commuting so much more convenient.", "positive"),
    ("Exceptional craftsmanship, every detail has been carefully considered.", "positive"),
    ("The spa treatment was deeply relaxing and rejuvenating.", "positive"),
    ("I highly appreciate the transparency shown by the management.", "positive"),
    ("The wildlife sanctuary was awe-inspiring and well-managed.", "positive"),
    ("The new collection is stylish, modern, and very affordable.", "positive"),
    ("I am overjoyed by the test results, hard work really pays off.", "positive"),
    ("The podcast episode was engaging, informative, and entertaining.", "positive"),
    ("The renovation turned out even better than we had imagined.", "positive"),


    # ───────────── NEGATIVE ─────────────
    ("This is the worst product I have ever bought, total waste of money.", "negative"),
    ("The movie was boring and the plot made absolutely no sense.", "negative"),
    ("Customer service was rude and completely unhelpful.", "negative"),
    ("I am extremely disappointed with the quality of this item.", "negative"),
    ("The restaurant food was cold, tasteless, and overpriced.", "negative"),
    ("My order arrived broken and the return process is a nightmare.", "negative"),
    ("This book was a chore to read, I could not finish it.", "negative"),
    ("The project was delivered late and full of errors.", "negative"),
    ("The concert was terrible, poor sound quality and bad organisation.", "negative"),
    ("The hotel was dirty, noisy, and far below the advertised standard.", "negative"),
    ("The app keeps crashing every time I try to open it.", "negative"),
    ("I regret buying this, it stopped working after two days.", "negative"),
    ("The performance was embarrassing and deeply disappointing.", "negative"),
    ("I feel let down by the lack of support from the organisation.", "negative"),
    ("The traffic was horrendous and my entire day was ruined.", "negative"),
    ("The doctor was dismissive and did not listen to my concerns.", "negative"),
    ("The road trip was a disaster, the car broke down twice.", "negative"),
    ("The workshop was poorly planned and a waste of everyone's time.", "negative"),
    ("The ride was terrifying and my children were upset all day.", "negative"),
    ("This coffee is bitter and completely undrinkable.", "negative"),
    ("The rejection letter was devastating after all that hard work.", "negative"),
    ("I am frustrated with the lack of progress in my fitness goals.", "negative"),
    ("The software update broke everything that was working before.", "negative"),
    ("The venue was chaotic, the staff were rude and disorganised.", "negative"),
    ("The fundraiser was a failure, nobody showed up to support it.", "negative"),
    ("The storyline was predictable and the acting was terrible.", "negative"),
    ("The lecture was confusing, monotone, and impossible to follow.", "negative"),
    ("The support team never responded to my emails or calls.", "negative"),
    ("The playground equipment is broken and unsafe for children.", "negative"),
    ("The training was irrelevant, outdated, and badly presented.", "negative"),
    ("The artwork was amateurish and not worth the high asking price.", "negative"),
    ("Poor value for money, I expected so much better than this.", "negative"),
    ("The new policy has harmed staff morale and productivity.", "negative"),
    ("I left the conference feeling drained and thoroughly uninspired.", "negative"),
    ("The garden is completely overgrown and has been neglected.", "negative"),
    ("The research was flawed and the conclusions were misleading.", "negative"),
    ("The volunteers were disorganised and the event was a mess.", "negative"),
    ("My delivery is weeks late and customer service won't help.", "negative"),
    ("The stand-up routine was offensive and not funny at all.", "negative"),
    ("The construction is an eyesore and has ruined the neighbourhood.", "negative"),
    ("The mentorship program was unhelpful and poorly structured.", "negative"),
    ("The recipe was a disaster, inedible and wasteful.", "negative"),
    ("The gym equipment is dangerous and frequently out of order.", "negative"),
    ("I am ashamed of how the students have been treated this year.", "negative"),
    ("The documentary was misleading, biased, and poorly researched.", "negative"),
    ("The new bus route is slower and more inconvenient than before.", "negative"),
    ("Shoddy workmanship, everything fell apart within a week.", "negative"),
    ("The so-called spa was dirty, and the staff were unfriendly.", "negative"),
    ("The management has been opaque and dishonest with employees.", "negative"),
    ("The sanctuary was overcrowded and the animals seemed distressed.", "negative"),
    ("The collection is overpriced, outdated, and poorly made.", "negative"),
    ("I am devastated by the results after months of preparation.", "negative"),
    ("The podcast was rambling, disorganised, and hard to follow.", "negative"),
    ("The renovation looks sloppy, and several issues remain unfixed.", "negative"),


    # ───────────── NEUTRAL ─────────────
    ("The package arrived on Tuesday as scheduled.", "neutral"),
    ("The film is two hours and fifteen minutes long.", "neutral"),
    ("The store opens at nine in the morning on weekdays.", "neutral"),
    ("The report covers data from January through December.", "neutral"),
    ("The meeting has been rescheduled to Thursday afternoon.", "neutral"),
    ("The product is available in three different colour options.", "neutral"),
    ("The temperature today is around twenty degrees Celsius.", "neutral"),
    ("The company was founded in nineteen ninety-eight.", "neutral"),
    ("The new model features a larger screen and longer battery life.", "neutral"),
    ("The book has three hundred and forty-two pages in total.", "neutral"),
    ("The flight departs at six-fifteen in the morning.", "neutral"),
    ("The survey collected responses from five hundred participants.", "neutral"),
    ("The office is located on the fourth floor of the building.", "neutral"),
    ("The course runs for a period of twelve weeks.", "neutral"),
    ("The software requires at least eight gigabytes of RAM.", "neutral"),
    ("The station is approximately two kilometres from here.", "neutral"),
    ("The annual report will be published next quarter.", "neutral"),
    ("The event attracted roughly three hundred attendees.", "neutral"),
    ("The product weighs four hundred grams without packaging.", "neutral"),
    ("The committee will meet again next month to review progress.", "neutral"),
    ("The application deadline is the fifteenth of next month.", "neutral"),
    ("The restaurant is open seven days a week.", "neutral"),
    ("The project budget is set at fifty thousand dollars.", "neutral"),
    ("The document has been updated to reflect the latest changes.", "neutral"),
    ("The train runs every thirty minutes during peak hours.", "neutral"),
    ("The team consists of twelve members from five departments.", "neutral"),
    ("The library closes at eight o'clock on weekday evenings.", "neutral"),
    ("The experiment used a sample size of two hundred subjects.", "neutral"),
    ("The vehicle has a fuel efficiency of fifteen kilometres per litre.", "neutral"),
    ("The university offers both full-time and part-time enrolment.", "neutral"),
    ("The policy will take effect from the first of next month.", "neutral"),
    ("The manufacturer provides a two-year warranty on the product.", "neutral"),
    ("The website was last updated three weeks ago.", "neutral"),
    ("The stadium has a maximum capacity of forty-five thousand.", "neutral"),
    ("The prescription should be taken twice daily with meals.", "neutral"),
    ("The factory produces approximately one thousand units per day.", "neutral"),
    ("The user manual is available for download on the website.", "neutral"),
    ("The conference is scheduled to run for three days.", "neutral"),
    ("The system backs up data automatically every twenty-four hours.", "neutral"),
    ("The building has ten floors and two underground car parks.", "neutral"),
    ("The membership fee is renewed on an annual basis.", "neutral"),
    ("The programme is broadcast every Wednesday at nine PM.", "neutral"),
    ("The transaction was processed and confirmed successfully.", "neutral"),
    ("The office will be closed on all public holidays.", "neutral"),
    ("The update introduced several minor bug fixes and improvements.", "neutral"),
    ("The candidate submitted all required documents by the deadline.", "neutral"),
    ("The road is closed for maintenance until further notice.", "neutral"),
    ("The agenda for the meeting has been circulated to all members.", "neutral"),
    ("The device is compatible with both iOS and Android systems.", "neutral"),
    ("The results will be announced at the end of the month.", "neutral"),
    ("The contract is valid for a period of one year from signing.", "neutral"),
    ("The data is stored securely in an encrypted cloud environment.", "neutral"),
]
print(f"Total dataset size: {len(RAW_DATA)} examples")
for label in ["positive", "negative", "neutral"]:
    count = sum(1 for _, l in RAW_DATA if l == label)
    print(f"  {label:>10}: {count}")



Total dataset size: 160 examples
    positive: 54
    negative: 54
     neutral: 52


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5
WARMUP_RATIO = 0.1

In [ ]:
texts = [t for t, _ in RAW_DATA]
labels = [LABEL2ID[l] for _, l in RAW_DATA]

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.25, random_state=SEED, stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

120 20 20


In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

train_ds = SentimentDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = SentimentDataset(X_val, y_val, tokenizer, MAX_LEN)
test_ds  = SentimentDataset(X_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(DEVICE)

[transformers] loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
[transformers] Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

In [ ]:
def train_epoch():
    model.train()
    total_loss, preds_all, labels_all = 0, [], []

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds_all += torch.argmax(outputs.logits, 1).cpu().tolist()
        labels_all += labels.cpu().tolist()

    acc = accuracy_score(labels_all, preds_all)
    return total_loss / len(train_loader), acc

In [ ]:
def evaluate(loader):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)

            total_loss += outputs.loss.item()
            preds_all += torch.argmax(outputs.logits, 1).cpu().tolist()
            labels_all += labels.cpu().tolist()

    acc = accuracy_score(labels_all, preds_all)
    return total_loss / len(loader), acc, preds_all, labels_all

In [ ]:
best_acc = 0
best_path = "best_model.pt"

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch()
    val_loss, val_acc, _, _ = evaluate(val_loader)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Train Loss:", train_loss, "Train Acc:", train_acc)
    print("Val Loss:", val_loss, "Val Acc:", val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), best_path)


Epoch 1/5
Train Loss: 1.118599310517311 Train Acc: 0.36666666666666664
Val Loss: 1.062016248703003 Val Acc: 0.45

Epoch 2/5
Train Loss: 0.9828322902321815 Train Acc: 0.575
Val Loss: 1.0091566443443298 Val Acc: 0.65

Epoch 3/5
Train Loss: 0.9012745842337608 Train Acc: 0.675
Val Loss: 0.9107162058353424 Val Acc: 0.65

Epoch 4/5
Train Loss: 0.8031649217009544 Train Acc: 0.725
Val Loss: 0.8650736808776855 Val Acc: 0.7

Epoch 5/5
Train Loss: 0.7945810258388519 Train Acc: 0.75
Val Loss: 0.8579867780208588 Val Acc: 0.75


In [ ]:
model.load_state_dict(torch.load(best_path))

test_loss, test_acc, preds, labels = evaluate(test_loader)

print("\nTest Accuracy:", test_acc)
print(classification_report(labels, preds, target_names=["negative", "neutral", "positive"]))
print(confusion_matrix(labels, preds))


Test Accuracy: 0.85
              precision    recall  f1-score   support

    negative       0.78      1.00      0.88         7
     neutral       0.88      1.00      0.93         7
    positive       1.00      0.50      0.67         6

    accuracy                           0.85        20
   macro avg       0.88      0.83      0.83        20
weighted avg       0.88      0.85      0.83        20

[[7 0 0]
 [0 7 0]
 [2 1 3]]


In [ ]:
def predict_sentiment(text):
    model.eval()

    enc = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    with torch.no_grad():
        outputs = model(
            input_ids=enc["input_ids"].to(DEVICE),
            attention_mask=enc["attention_mask"].to(DEVICE),
        )

    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred = torch.argmax(probs).item()

    return {
        "label": ID2LABEL[pred],
        "confidence": {
            ID2LABEL[i]: float(probs[i]) for i in range(3)
        }
    }

In [ ]:
def sample_predictions(n=15):
    idxs = random.sample(range(len(X_test)), n)

    for i in idxs:
        text = X_test[i]
        true = ID2LABEL[y_test[i]]
        pred = predict_sentiment(text)

        print("\nText:", text)
        print("True:", true)
        print("Pred:", pred["label"])
        print("Confidence:", pred["confidence"])

sample_predictions(15)


Text: The building has ten floors and two underground car parks.
True: neutral
Pred: neutral
Confidence: {'negative': 0.26909562945365906, 'neutral': 0.5425488948822021, 'positive': 0.18835550546646118}

Text: The road is closed for maintenance until further notice.
True: neutral
Pred: neutral
Confidence: {'negative': 0.1977137178182602, 'neutral': 0.6113163828849792, 'positive': 0.19096989929676056}

Text: The new policy has greatly benefited employees across the board.
True: positive
Pred: neutral
Confidence: {'negative': 0.31949612498283386, 'neutral': 0.43733471632003784, 'positive': 0.2431691288948059}

Text: The office will be closed on all public holidays.
True: neutral
Pred: neutral
Confidence: {'negative': 0.21305452287197113, 'neutral': 0.595751166343689, 'positive': 0.1911943256855011}

Text: The transaction was processed and confirmed successfully.
True: neutral
Pred: neutral
Confidence: {'negative': 0.2556185722351074, 'neutral': 0.5206965208053589, 'positive': 0.22368495